# Externalization through graph-mode quantization

This notebook demos the new `mark_for_externalization` API in `coreai-torch`, which lets graph-mode quantization tools (e.g. `coreai-opt`) produce composite-decl'd `coreai.graph`s for things like RMSNorm, Attention, etc.

The key idea: **mark the boundary BEFORE tracing** by patching matching submodules' `forward` with a `torch.library.custom_op`. The custom op is opaque to every downstream FX pass — `torch.export`, decompositions, q/dq insertion all walk past it — so the boundary survives intact and we never need to reconstruct module identity from `nn_module_stack` metadata.

Pipeline:

```
Stage 0: nn.Module
    │  mark_for_externalization()              ── Phase 1 ──
    ▼
Stage 1: nn.Module with patched .forward = custom_op
    │  user's export / quantization pipeline    (Phase 2)
    ▼
Stage 2: ExportedProgram (parent, possibly quantized) with opaque
         custom_op call sites at every externalization boundary
    │  TorchConverter().add_exported_program(ep, externalize=markers)
    │  to_coreai()
    ▼
Stage 3: AIProgram with
         - coreai.graph @main (with coreai.invoke at every boundary)
         - coreai.graph private noinline @<sub> with composite_decl, per boundary
```

In [ ]:
import torch
import torch.nn as nn
from coreai_torch import (
    TorchConverter,
    ExternalizeSpec,
    mark_for_externalization,
    get_decomp_table,
)

torch.manual_seed(0)

## A tiny model with a marker-friendly submodule

Note `forward` has type annotations — `torch.library.custom_op` infers its schema from them, so this is a hard requirement on any submodule you want to externalize.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        v = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(v + self.eps) * self.weight


class Block(nn.Module):
    def __init__(self, dim: int = 16):
        super().__init__()
        self.norm = RMSNorm(dim)
        self.lin = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lin(self.norm(x))


model = Block().eval()
sample = (torch.randn(2, 16),)
model

## Stage 0: original module, no patching

In [ ]:
print("forward: ", model.norm.forward)
print("_original_forward present:", hasattr(model.norm, "_original_forward"))

## Stage 1: enter `mark_for_externalization`

Walks `named_modules`, finds every `RMSNorm`, replaces its `forward` with a `torch.library.custom_op`. The model is now a hybrid: real submodule attrs + a patched dispatcher. It still runs end-to-end.

In [ ]:
markers_ctx = mark_for_externalization(
    model,
    [
        ExternalizeSpec(
            target_class=RMSNorm,
            composite_op_name="rms_norm",
            composite_attrs=["eps"],
        ),
    ],
)
markers = markers_ctx.__enter__()  # for cell-by-cell demo; in real code use `with`

print("forward: ", model.norm.forward)
print("_original_forward present:", hasattr(model.norm, "_original_forward"))
print("_externalize_config:      ", model.norm._externalize_config)
print()
print("forward still callable:   ", model(*sample).shape)

## Stage 2: user's pipeline runs `torch.export.export`

**This is what `coreai-opt`'s graph-mode quantizer does internally** in `prepare()`. We do it directly here so the demo is self-contained.

Look at the resulting graph: the entire RMSNorm body collapses to **one** `coreai_torch_externalize::norm` call_function node. `torch.export` does not recurse into custom ops.

In [ ]:
ep = torch.export.export(model, args=sample).run_decompositions(get_decomp_table())
# Keep a separate clean copy for the actual conversion in Stage 3 — cell 12
# below mutates `ep` for illustration.
ep_clean = torch.export.export(model, args=sample).run_decompositions(get_decomp_table())
ep.graph_module.print_readable(print_output=False)

In [ ]:
# Confirm the boundary is one opaque node:
for n in ep.graph.nodes:
    if n.op == "call_function":
        print(f"  {n.op:14s} {n.name:20s} target={n.target}")

## Stage 2b: simulate what graph-mode quantization adds

Real `coreai-opt` would now insert observers/q-dq pairs around `aten.linear`, `aten.matmul`, etc. — but it has no annotation pattern for `coreai_torch_externalize::norm`, so that node passes through untouched, and the q/dq pairs naturally land at its **boundary**.

We simulate this by hand: insert a fake-quant-style `q→dq` pair on the input to `aten.linear` and on the linear weight. The point is to show the converter still finds the custom-op boundary even when the surrounding ops have been rewritten.

**To use real `coreai-opt` instead, replace this cell with:**

```python
from coreai_opt.quantization import Quantizer, QuantizationConfig, ExecutionMode
quantizer = Quantizer(model, config, execution_mode=ExecutionMode.GRAPH)
prepared = quantizer.prepare(sample)
# (calibrate if you have activation quant)
ep = quantizer.finalize(backend=ExportBackend.CoreAI_EP)
```

and skip the manual rewrite below.

In [ ]:
# Manual q/dq simulation — strictly for demoing FX-graph survival of the boundary.
# After get_decomp_table, linear becomes permute + addmm; we put a fake q/dq pair
# on addmm's activation input so it sits right next to the custom-op call site.
import torch.fx as fx
import torch.ao.quantization.fx._decomposed  # noqa: F401  registers torch.ops.quantized_decomposed

gm = ep.graph_module
graph = gm.graph

matmul_node = next(
    n for n in graph.nodes
    if n.op == "call_function"
    and any(s in str(n.target) for s in ("addmm", "mm.default", "linear"))
)
# addmm signature is addmm(bias, lhs, rhs); lhs is the activation we fake-quant.
args = list(matmul_node.args)
act_idx = 1 if "addmm" in str(matmul_node.target) else 0
act = args[act_idx]

with graph.inserting_before(matmul_node):
    x_q = graph.call_function(
        torch.ops.quantized_decomposed.quantize_per_tensor.default,
        args=(act, 0.05, 0, -128, 127, torch.int8),
    )
    x_dq = graph.call_function(
        torch.ops.quantized_decomposed.dequantize_per_tensor.default,
        args=(x_q, 0.05, 0, -128, 127, torch.int8),
    )
args[act_idx] = x_dq
matmul_node.args = tuple(args)

graph.lint()
gm.recompile()
gm.print_readable(print_output=False)

Notice: `coreai_torch_externalize::norm` is still right there next to the q/dq nodes. That's the entire point — its boundary is real FX nodes, not metadata, so quant passes can rewrite *around* it but not *through* it.

## Stage 3: convert with `add_exported_program(..., externalize=markers)`

The converter sees `externalize=markers`, walks the still-patched model to find marked submodules, finds their custom-op call sites in the EP, restores each `_original_forward`, sub-exports each submodule on its own (giving the composite body), and emits:

- one `coreai.graph private noinline @<sub>` per call site, carrying `composite_decl = "rms_norm"` and the `eps` attr;
- one `coreai.invoke @<sub>` in the parent graph at each former custom-op call site;
- normal lowering for everything else (including q/dq nodes if the converter has handlers, or you can register your own via `register_torch_lowering`).

In [ ]:
# We use the un-mutated `ep_clean` here (the converter has no lowering for the
# fake q/dq ops we hand-inserted in cell 12 — real `coreai-opt` would provide
# those lowerings via its CoreAI_EP backend). The externalize plumbing behaves
# identically with or without the surrounding q/dq.
program = (
    TorchConverter()
    .add_exported_program(ep_clean, externalize=markers)
    .to_coreai()
)
print(str(program))

Things to look for in the IR above:

- **`coreai.graph private noinline @norm_<hash>`** — the externalized RMSNorm body, lowered from the *original* `forward` (the patch was restored before sub-export).
- **`composite_decl = #coreai.composite_declaration<"rms_norm" = {... op_attrs = {eps = 9.99999974E-6 ...} ...}>`** — the `composite_op_name` and `composite_attrs` from `ExternalizeSpec` flow into the IR attribute.
- **`coreai.graph @main`** — the parent program, with **`coreai.invoke @norm_<hash>(%arg0)`** at the call site that used to be the custom-op call_function.
- **No `pow / mean / rsqrt / mul`** in `@main` — they're only in `@norm_…`. The boundary held.

## Stage 4: exit the markers context — model fully restored

In real code this happens automatically when the `with` block exits. Here we did it cell-by-cell, so we exit explicitly.

In [ ]:
markers_ctx.__exit__(None, None, None)

print("forward: ", model.norm.forward)
print("_original_forward present:", hasattr(model.norm, "_original_forward"))
print("_externalize_config present:", hasattr(model.norm, "_externalize_config"))
print()
print("model still works:", model(*sample).shape)

## End-to-end, the way `coreai-opt` would call it

Putting it all together — the canonical usage pattern, with the user's `with` block scoping the patch:

In [ ]:
model = Block().eval()
sample = (torch.randn(2, 16),)

with mark_for_externalization(
    model,
    [ExternalizeSpec(RMSNorm, composite_op_name="rms_norm", composite_attrs=["eps"])],
) as markers:
    # === replace this block with: prepared = quantizer.prepare(sample); calibrate(); ep = quantizer.finalize() ===
    ep = torch.export.export(model, args=sample).run_decompositions(get_decomp_table())
    # =====================================================================================================

    program = (
        TorchConverter()
        .add_exported_program(ep, externalize=markers)
        .to_coreai()
    )

# Outside the `with`: model is restored, program is finalized.
assert not hasattr(model.norm, "_original_forward")
print("OK — model restored, IR contains:")
for line in str(program).splitlines():
    if "coreai.graph" in line or "coreai.invoke" in line or "composite_decl" in line:
        print(" ", line.strip()[:140])